# train

Full fine-tune of one (model size, seed) point of the scaling study. Trains only — evaluation
lives in `evaluation.ipynb`.

**Output:** `PROJECT_ROOT/models/<model>_seed<seed>/` (weights, tokenizer, `train_log.csv`,
`run_config.json`).

**To run the sweep:** change `MODEL` (size axis) and `SEED` (3 seeds per size). Keep everything
in the *frozen study constants* block identical across every run — any drift turns the scaling
curve into noise.

In [ ]:
# === config ===
import csv, json, os, time, math, random
from pathlib import Path
import torch
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          get_cosine_schedule_with_warmup, set_seed)
from torch.utils.data import DataLoader
from dotenv import load_dotenv

# ---- FROZEN study constants: IDENTICAL across every size and seed ----
REPO        = "notoh/exebench_llvm_o3"
EPOCHS      = 3            # >1: one epoch undertrains generalization (validated on smoke run)
LR          = 2e-5         # full-FT LR
GRAD_ACCUM  = 8
WARMUP      = 10
BATCH       = 1            # raise on GPU if memory allows; step count auto-recomputes
MAX_LEN     = 7600
PROMPT_TMPL = "Compile the following C to LLVM IR at -O3:\n{c}\n---\n"

PROJECT_ROOT = Path("../..").resolve()
load_dotenv(PROJECT_ROOT / ".env")
MODEL    = os.environ["BASEMODEL"]
SEED     = int(os.environ["SEED"])
REVISION = os.environ["REVISION"]

set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device      :", device)
print("project root:", PROJECT_ROOT)
print(f"MODEL={MODEL}  SEED={SEED}  EPOCHS={EPOCHS}")

In [ ]:
# === load tokenizer, model, data ===
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

dtype = torch.bfloat16 if device == "cuda" else torch.float32
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=dtype)
model.to(device)
model.gradient_checkpointing_enable()
model.config.use_cache = False        # required with gradient checkpointing

train_ds = load_dataset(REPO, revision=REVISION)["train"]
print(f"train examples: {len(train_ds)}")

RUN_NAME = f"{MODEL.split('/')[-1]}_seed{SEED}_{len(train_ds)}"
OUT_DIR  = PROJECT_ROOT / "models" / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("output dir  :", OUT_DIR)

In [ ]:
# === collation + completion-only loss masking ===
EOS_ID = tok.eos_token_id

def build_example(rec):
    prompt = PROMPT_TMPL.format(c=rec["c_source"])
    p_ids  = tok(prompt, add_special_tokens=False)["input_ids"]
    # append the EOS *id* directly (robust: no dependence on string re-tokenization)
    t_ids  = tok(rec["o3_ir"], add_special_tokens=False)["input_ids"] + [EOS_ID]
    input_ids = (p_ids + t_ids)[:MAX_LEN]
    labels    = ([-100]*len(p_ids) + t_ids)[:MAX_LEN]   # mask prompt, grade IR (+EOS)
    return input_ids, labels

def collate(batch):
    rows = [build_example(r) for r in batch]
    maxlen = max(len(x[0]) for x in rows)
    pad = tok.pad_token_id
    input_ids, labels, attn = [], [], []
    for ids, lab in rows:
        n = maxlen - len(ids)
        input_ids.append(ids + [pad]*n)
        labels.append(lab + [-100]*n)
        attn.append([1]*len(ids) + [0]*n)
    return {"input_ids": torch.tensor(input_ids),
            "labels": torch.tensor(labels),
            "attention_mask": torch.tensor(attn)}

# sanity: graded tokens > 0; flag MAX_LEN truncation (would cut the EOS the model needs to learn)
for idx in random.Random(SEED).sample(range(len(train_ds)), min(5, len(train_ds))):
    ids, lab = build_example(train_ds[idx])
    graded = sum(1 for x in lab if x != -100)
    assert graded > 0, f"row {idx}: zero graded tokens (empty/truncated IR?)"
    print(f"row {idx:>4}  graded {graded:>5}"
          f"  {'[TRUNCATED at MAX_LEN -> EOS lost]' if len(ids) >= MAX_LEN else ''}")

In [ ]:
# === train loop ===
train_loader    = DataLoader(train_ds, batch_size=BATCH, shuffle=True, collate_fn=collate)
num_batches     = len(train_loader)
steps_per_epoch = math.ceil(num_batches / GRAD_ACCUM)
total_steps     = steps_per_epoch * EPOCHS
print(f"num_batches={num_batches}  steps/epoch={steps_per_epoch}  epochs={EPOCHS}  total_steps={total_steps}")

opt   = torch.optim.AdamW(model.parameters(), lr=LR)
sched = get_cosine_schedule_with_warmup(opt, WARMUP, max(total_steps, 1))

logf   = open(OUT_DIR / "train_log.csv", "w", newline="")
writer = csv.writer(logf); writer.writerow(["step", "epoch", "loss", "lr", "secs"])

model.train()
step, t0, window_loss = 0, time.time(), 0.0
opt.zero_grad()
for epoch in range(EPOCHS):
    for i, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        win_start = (i // GRAD_ACCUM) * GRAD_ACCUM
        win_size  = min(GRAD_ACCUM, num_batches - win_start)   # correct partial-tail normalization
        loss = model(**batch).loss / win_size
        loss.backward()
        window_loss += loss.item()
        if ((i + 1) % GRAD_ACCUM == 0) or ((i + 1) == num_batches):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
            step += 1
            secs = round(time.time() - t0, 1)
            if step == 1 or step % 10 == 0 or step == total_steps:
                print(f"step {step}/{total_steps}  ep{epoch}  loss {window_loss:.4f}  "
                      f"lr {sched.get_last_lr()[0]:.2e}  {secs}s")
            writer.writerow([step, epoch, window_loss, sched.get_last_lr()[0], secs]); logf.flush()
            window_loss = 0.0
logf.close()

# save weights + tokenizer + a run_config the eval notebook reads back (keeps prompt/seed in sync)
model.save_pretrained(OUT_DIR); tok.save_pretrained(OUT_DIR)
run_config = {"model": MODEL, "seed": SEED, "epochs": EPOCHS, "lr": LR,
              "grad_accum": GRAD_ACCUM, "warmup": WARMUP, "batch": BATCH,
              "max_len": MAX_LEN, "prompt_tmpl": PROMPT_TMPL,
              "repo": REPO, "revision": REVISION, "total_steps": total_steps,
              "dataset_size": len(train_ds)}
(OUT_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2))
print("saved ->", OUT_DIR)